# Observation Probability Controls — Llama 3.2 3B

In [1]:
import os
os.environ['HF_HOME'] = '/workspace/hf_cache'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import gc
import numba
from tqdm import tqdm
from sklearn.model_selection import train_test_split

sns.set_context('notebook')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
np.random.seed(42); torch.manual_seed(42)

SEQ_LEN = 20_000
PROBE_START = 15_000
N_SEEDS = 10
TRAIN_FRAC = 0.2
# LAYERS set after model load
CHUNK_SIZE = 4096

RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)


## Model

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'meta-llama/Llama-3.2-3B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
    attn_implementation='sdpa', device_map='auto')
model.eval()

N_LAYERS = len(model.model.layers)
LAYERS = list(range(N_LAYERS))
print(f'Model: {MODEL_NAME}')
print(f'Layers: {N_LAYERS}, hidden_size: {model.config.hidden_size}')

TOKEN_NAMES_2 = np.array(['F', 'Q'])
TOKEN_NAMES_3 = np.array(['F', 'Q', 'V'])
TOK_IDS_2 = [tokenizer.encode(f' {n}', add_special_tokens=False)[-1] for n in TOKEN_NAMES_2]
TOK_IDS_3 = [tokenizer.encode(f' {n}', add_special_tokens=False)[-1] for n in TOKEN_NAMES_3]
print(f'2-token IDs: {dict(zip(TOKEN_NAMES_2, TOK_IDS_2))}')
print(f'3-token IDs: {dict(zip(TOKEN_NAMES_3, TOK_IDS_3))}')

# Verify single-token encoding
for name, tid in zip(TOKEN_NAMES_2, TOK_IDS_2):
    decoded = tokenizer.decode([tid])
    print(f'  {name} -> id={tid} -> decoded="{decoded}"')
for name, tid in zip(TOKEN_NAMES_3, TOK_IDS_3):
    decoded = tokenizer.decode([tid])
    print(f'  {name} -> id={tid} -> decoded="{decoded}"')


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Model: meta-llama/Llama-3.2-3B
Layers: 28, hidden_size: 3072
2-token IDs: {np.str_('F'): 435, np.str_('Q'): 1229}
3-token IDs: {np.str_('F'): 435, np.str_('Q'): 1229, np.str_('V'): 650}
  F -> id=435 -> decoded=" F"
  Q -> id=1229 -> decoded=" Q"
  F -> id=435 -> decoded=" F"
  Q -> id=1229 -> decoded=" Q"
  V -> id=650 -> decoded=" V"


## Infrastructure

In [3]:
def stationary_distribution(T_matrices):
    T_full = sum(T_matrices)
    eigvals, eigvecs = np.linalg.eig(T_full.T)
    idx = np.argmin(np.abs(eigvals - 1.0))
    pi = np.real(eigvecs[:, idx])
    return pi / pi.sum()

def sample_hmm_sequence(T_matrices, pi, seq_len, seed=None):
    rng = np.random.default_rng(seed)
    n_states, n_tokens = len(pi), len(T_matrices)
    state = rng.choice(n_states, p=pi)
    tokens = []
    for _ in range(seq_len):
        tp = np.array([T_matrices[z][state].sum() for z in range(n_tokens)])
        tp /= tp.sum()
        z = rng.choice(n_tokens, p=tp)
        tokens.append(z)
        nsp = T_matrices[z][state] / T_matrices[z][state].sum()
        state = rng.choice(n_states, p=nsp)
    return np.array(tokens)

def tokens_to_prompt(tokens, token_names, sep=' '):
    return sep + sep.join(token_names[t] for t in tokens)

def tokenize_prompt(prompt):
    return tokenizer.encode(prompt, return_tensors='pt', truncation=False)

def match_positions(input_ids, tok_ids):
    ids = input_ids[0].cpu().numpy()
    tok_id_set = {tid: zi for zi, tid in enumerate(tok_ids)}
    pos, tok = [], []
    for i, tid in enumerate(ids):
        if tid in tok_id_set:
            pos.append(i); tok.append(tok_id_set[tid])
    return np.array(pos), np.array(tok)

@numba.njit(cache=True)
def full_bayesian_beliefs_numba(tokens, T_stack, pi):
    n = len(tokens)
    n_states = len(pi)
    beliefs = np.zeros((n, n_states))
    b = pi.copy()
    for t in range(n):
        b = b @ T_stack[tokens[t]]
        s = 0.0
        for j in range(n_states):
            s += b[j]
        if s > 0:
            for j in range(n_states):
                b[j] /= s
        for j in range(n_states):
            beliefs[t, j] = b[j]
    return beliefs

def extract_and_probe_multi(input_ids, pos_indices, n_matched, targets_dict, seed):
    """Forward pass with hooks → R² for all layers, multiple targets.
    
    targets_dict: {'real': y_array, 'shuffle': y_array, 'random': y_array}
    Returns: {target_name: {layer: r2}}
    """
    seq_len = input_ids.shape[1]
    past_kv = None

    late_mask = np.arange(n_matched) >= PROBE_START
    late_model_pos = pos_indices[late_mask]
    n_late = int(late_mask.sum())

    acts = {l: [] for l in LAYERS}

    first_hooked_chunk = None
    for start in range(0, seq_len, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, seq_len)
        chunk = input_ids[:, start:end].to(device)
        need_hooks = (end > PROBE_START)
        hooks = []
        chunk_acts = {}

        if need_hooks:
            if first_hooked_chunk is None:
                first_hooked_chunk = start
            for l in LAYERS:
                def make_hook(li):
                    def fn(module, inp, out):
                        h = out[0] if isinstance(out, tuple) else out
                        chunk_acts[li] = h[0]
                    return fn
                hooks.append(model.model.layers[l].register_forward_hook(make_hook(l)))

        with torch.no_grad():
            out = model.model(chunk, past_key_values=past_kv, use_cache=True)

        for h in hooks:
            h.remove()

        if need_hooks:
            chunk_pos = np.arange(start, end)
            needed = np.isin(chunk_pos, late_model_pos)
            if needed.any():
                idx = torch.tensor(np.where(needed)[0], device=device)
                for l in LAYERS:
                    acts[l].append(chunk_acts[l][idx])

        past_kv = out.past_key_values
        del out, chunk_acts; torch.cuda.empty_cache()

    del past_kv; torch.cuda.empty_cache()

    # Train/test split (same for all targets)
    idx_tr, idx_te = train_test_split(
        np.arange(n_late), train_size=TRAIN_FRAC, random_state=seed)

    # Precompute Y tensors for all targets
    Y_targets = {}
    for tname, y in targets_dict.items():
        Y_targets[tname] = (
            torch.tensor(y[idx_tr], device=device, dtype=torch.float32),
            torch.tensor(y[idx_te], device=device, dtype=torch.float32),
        )

    # Probe each layer
    results = {tname: {} for tname in targets_dict}
    for l in LAYERS:
        X = torch.cat(acts[l], dim=0).float()
        X_tr = X[idx_tr]
        X_te = X[idx_te]
        P = torch.linalg.pinv(X_tr)  # shared across targets

        for tname in targets_dict:
            Y_tr, Y_te = Y_targets[tname]
            W = P @ Y_tr
            pred = X_te @ W
            ss_res = ((Y_te - pred) ** 2).sum().item()
            ss_tot = ((Y_te - Y_te.mean(0)) ** 2).sum().item()
            results[tname][l] = 1.0 - ss_res / ss_tot

        del X, X_tr, X_te, P

    del acts, Y_targets; torch.cuda.empty_cache()
    return results


def next_token_probs(beliefs, T_matrices):
    """Compute next-token probabilities from beliefs."""
    n_tok = len(T_matrices)
    probs = np.zeros((len(beliefs), n_tok))
    for k in range(n_tok):
        probs[:, k] = (beliefs @ T_matrices[k]).sum(axis=1)
    return probs

# Warmup numba
_d = np.random.rand(3, 4, 4); _p = np.array([0.25, 0.25, 0.25, 0.25])
_ = full_bayesian_beliefs_numba(np.array([0, 1, 2], dtype=np.int64), _d, _p)
del _d, _p
print('Infrastructure loaded. Numba compiled.')


Infrastructure loaded. Numba compiled.


## HMM Definitions

In [4]:
# ===== Mess3 (3 tokens: A, B, C) =====

def mess3_matrices(a, x):
    b = (1 - a) / 2
    y = 1 - 2 * x
    ay, bx, by, ax = a*y, b*x, b*y, a*x
    return [
        np.array([[ay, bx, bx], [ax, by, bx], [ax, bx, by]]),
        np.array([[by, ax, bx], [bx, ay, bx], [bx, ax, by]]),
        np.array([[by, bx, ax], [bx, by, ax], [bx, bx, ay]]),
    ]

# ===== Wing (2 tokens: F, Q) =====

def wing_matrices(x, y):
    b = (1 - x) / 2
    return [
        np.array([[0, b, 0], [0, y*x, 0.5*b], [b, 0, 0]]),
        np.array([[x, 0, b], [b, (1-y)*x, 0.5*b], [0, b, x]]),
    ]

# ===== Strata (2 tokens: F, Q) =====

def strata_matrices(a, t0, t1):
    b = (1 - a) / 2
    return [
        np.array([[t0*a, 0, 0], [0, t1*a, 0], [0, 0, 0]]),
        np.array([[(1-t0)*a, b, b], [b, (1-t1)*a, b], [b, b, a]]),
    ]

# ===== Arch (3 tokens: A, B, C) =====

def arch_matrices(a):
    b = (1 - a) / 3
    return [
        np.array([
            [0.8*a, 0, 0, 0],
            [0, 0.2*a, 0, 0],
            [0, 0, 0.4*a, 0],
            [0, 0, 0, 0.6*a],
        ]),
        np.array([
            [0, 0, 0, 0],
            [0, 0.4*a, 0, 0.4*b],
            [0, 0, 0.3*a, 0],
            [0, 0, 0, 0.16*a],
        ]),
        np.array([
            [0.2*a, b, b, b],
            [b, 0.4*a, b, 0.6*b],
            [b, b, 0.3*a, b],
            [b, b, b, 0.24*a],
        ]),
    ]

# ===== Spiral (2 tokens: F, Q) =====

def spiral_matrices(a):
    return [
        np.array([
            [0.2*a,       0,   0     ],
            [0,           0,   0     ],
            [0.25*(1-a),  0,   0.5*a ],
        ]),
        np.array([
            [0.8*a,       1-a, 0     ],
            [0,           a,   1-a   ],
            [0.75*(1-a),  0,   0.5*a ],
        ]),
    ]

print('All HMM functions defined.')


All HMM functions defined.


## HMM Registry

In [5]:
HMMS = {
    'Wing': {
        'fn': wing_matrices,
        'params': [(x, 0.4) for x in np.arange(0.90, 1.00, 0.01).round(2)],
        'label_fn': lambda p: f'x={p[0]}, y={p[1]}',
        'token_names': np.array(['F', 'Q']),
    },
    'Strata': {
        'fn': strata_matrices,
        'params': [(a, 0.38, 0.54) for a in np.arange(0.90, 1.00, 0.01).round(2)],
        'label_fn': lambda p: f'a={p[0]}, t0={p[1]}, t1={p[2]}',
        'token_names': np.array(['F', 'Q']),
    },
    'Arch': {
        'fn': arch_matrices,
        'params': [(a,) for a in np.arange(0.90, 1.00, 0.01).round(2)],
        'label_fn': lambda p: f'a={p[0]}',
        'token_names': np.array(['F', 'Q', 'V']),
    },
    'Mess3': {
        'fn': mess3_matrices,
        'params': [
            (0.005, 0.01), (0.005, 0.02), (0.01, 0.02), (0.05, 0.02), (0.10, 0.02),
            (0.60, 0.02), (0.70, 0.02), (0.80, 0.02), (0.85, 0.02), (0.90, 0.02),
        ],
        'label_fn': lambda p: f'a={p[0]}, x={p[1]}',
        'token_names': np.array(['F', 'Q', 'V']),
    },
}

for name, cfg in HMMS.items():
    T = cfg['fn'](*cfg['params'][0])
    print(f'{name:>8}: {len(cfg["params"])} params, {len(T)} tokens, {T[0].shape[0]} states')

    Wing: 10 params, 2 tokens, 3 states
  Strata: 10 params, 2 tokens, 3 states
    Arch: 10 params, 3 tokens, 4 states
   Mess3: 10 params, 3 tokens, 3 states


## Compute R² per layer (all HMMs)

In [ ]:
all_r2_rows = []

for hmm_name, cfg in HMMS.items():
    print(f'\n===== {hmm_name} =====')
    token_names = cfg['token_names']
    n_tok = len(token_names)
    tok_ids = [tokenizer.encode(f' {n}', add_special_tokens=False)[-1] for n in token_names]

    pbar = tqdm(total=len(cfg['params']) * N_SEEDS, desc=hmm_name)

    for param in cfg['params']:
        label = cfg['label_fn'](param)
        T_real = cfg['fn'](*param)
        T_stack = np.stack(T_real)
        pi_real = stationary_distribution(T_real)
        n_states = len(pi_real)

        # ---- Constant R²: ntp → beliefs and log_ntp → beliefs ----
        # Compute from pooled data (all seeds)
        all_beliefs_pool = []
        all_ntp_pool = []
        for seed in range(N_SEEDS):
            tokens = sample_hmm_sequence(T_real, pi_real, SEQ_LEN, seed=seed)
            beliefs = full_bayesian_beliefs_numba(tokens.astype(np.int64), T_stack, pi_real)
            ntp = next_token_probs(beliefs, T_real)
            all_beliefs_pool.append(beliefs[PROBE_START:])
            all_ntp_pool.append(ntp[PROBE_START:])

        b_pool = np.concatenate(all_beliefs_pool, axis=0)
        ntp_pool = np.concatenate(all_ntp_pool, axis=0)
        log_ntp_pool = np.log(ntp_pool + 1e-12)

        # ntp → beliefs
        X_ntp = np.hstack([ntp_pool, np.ones((len(ntp_pool), 1))])
        W_ntp = np.linalg.pinv(X_ntp) @ b_pool
        pred_ntp = X_ntp @ W_ntp
        ss_res = ((b_pool - pred_ntp) ** 2).sum()
        ss_tot = ((b_pool - b_pool.mean(0)) ** 2).sum()
        r2_ntp_to_beliefs = 1.0 - ss_res / ss_tot

        # log_ntp → beliefs
        X_log = np.hstack([log_ntp_pool, np.ones((len(log_ntp_pool), 1))])
        W_log = np.linalg.pinv(X_log) @ b_pool
        pred_log = X_log @ W_log
        ss_res = ((b_pool - pred_log) ** 2).sum()
        ss_tot = ((b_pool - b_pool.mean(0)) ** 2).sum()
        r2_logntp_to_beliefs = 1.0 - ss_res / ss_tot

        print(f'  {label}: R²(ntp→beliefs)={r2_ntp_to_beliefs:.4f}  R²(log_ntp→beliefs)={r2_logntp_to_beliefs:.4f}')
        del b_pool, ntp_pool, log_ntp_pool, all_beliefs_pool, all_ntp_pool

        for seed in range(N_SEEDS):
            tokens = sample_hmm_sequence(T_real, pi_real, SEQ_LEN, seed=seed)
            beliefs = full_bayesian_beliefs_numba(tokens.astype(np.int64), T_stack, pi_real)
            ntp = next_token_probs(beliefs, T_real)

            prompt = tokens_to_prompt(tokens, token_names)
            input_ids = tokenize_prompt(prompt)
            pos_indices, tok_at_pos = match_positions(input_ids, tok_ids)
            n_matched = min(len(tokens), len(pos_indices))

            n_late = n_matched - PROBE_START
            y_ntp = ntp[PROBE_START:n_matched]
            y_log_ntp = np.log(y_ntp + 1e-12)

            targets = {'act→ntp': y_ntp, 'act→log_ntp': y_log_ntp}
            results = extract_and_probe_multi(input_ids, pos_indices, n_matched, targets, seed)

            for tname, layer_r2 in results.items():
                for layer, r2 in layer_r2.items():
                    all_r2_rows.append({
                        'hmm': hmm_name, 'param': label,
                        'layer': layer, 'seed': seed,
                        'target': tname, 'R2': r2,
                    })

            # Add constant lines (same value for every layer, every seed)
            for layer in LAYERS:
                all_r2_rows.append({
                    'hmm': hmm_name, 'param': label,
                    'layer': layer, 'seed': seed,
                    'target': 'ntp→beliefs', 'R2': r2_ntp_to_beliefs,
                })
                all_r2_rows.append({
                    'hmm': hmm_name, 'param': label,
                    'layer': layer, 'seed': seed,
                    'target': 'log_ntp→beliefs', 'R2': r2_logntp_to_beliefs,
                })

            gc.collect(); torch.cuda.empty_cache()
            pbar.update(1)

    pbar.close()

    r2_df = pd.DataFrame(all_r2_rows)
    r2_df.to_csv(os.path.join(RESULTS_DIR, 'obsprob_llama32_3b.csv'), index=False)
    print(f'  Saved ({hmm_name} done, {len(r2_df)} rows)')

print(f'\nAll done.')



===== Wing =====


Wing:   0%|          | 0/100 [00:00<?, ?it/s]

  x=0.9, y=0.4: R²(ntp→beliefs)=0.8768  R²(log_ntp→beliefs)=0.8953


Wing:  10%|█         | 10/100 [02:28<21:49, 14.55s/it]

  x=0.91, y=0.4: R²(ntp→beliefs)=0.8786  R²(log_ntp→beliefs)=0.8948


Wing:  20%|██        | 20/100 [04:51<18:50, 14.14s/it]

  x=0.92, y=0.4: R²(ntp→beliefs)=0.8780  R²(log_ntp→beliefs)=0.8920


Wing:  30%|███       | 30/100 [07:14<15:29, 13.28s/it]

  x=0.93, y=0.4: R²(ntp→beliefs)=0.8778  R²(log_ntp→beliefs)=0.8894


Wing:  40%|████      | 40/100 [09:43<14:34, 14.58s/it]

## Load (after restart)

In [ ]:
r2_df = pd.read_csv(os.path.join(RESULTS_DIR, 'r2_qwen35_9b.csv'))
print(f'{len(r2_df)} rows, HMMs: {r2_df["hmm"].unique()}, targets: {r2_df["target"].unique()}')

## Plots: R² vs layer per HMM

In [ ]:
from matplotlib.gridspec import GridSpec

for hmm_name, grp in r2_df.groupby('hmm'):
    real = grp[grp['target'] == 'real']
    shuffle = grp[grp['target'] == 'shuffle']
    random = grp[grp['target'] == 'random']

    # Compute ranges for split
    real_stats = real.groupby('layer')['R2'].mean()
    shuf_stats = shuffle.groupby('layer')['R2'].mean()
    rand_stats = random.groupby('layer')['R2'].mean()
    ctrl_min = min(shuf_stats.min(), rand_stats.min()) - 0.02
    ctrl_max = max(shuf_stats.max(), rand_stats.max()) + 0.02
    real_min = max(0, real_stats.min() - 0.05)

    fig = plt.figure(figsize=(12, 7))
    gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)
    ax_top = fig.add_subplot(gs[0])
    ax_bot = fig.add_subplot(gs[1], sharex=ax_top)

    params = sorted(real['param'].unique())
    cmap = plt.cm.viridis(np.linspace(0.1, 0.9, len(params)))

    # Top: real R² per param
    for i, param in enumerate(params):
        sub = real[real['param'] == param]
        stats = sub.groupby('layer')['R2'].agg(['mean', 'sem']).reset_index()
        ax_top.plot(stats['layer'], stats['mean'], '-', color=cmap[i], lw=2, label=param)

    ax_top.set_ylim(real_min, 1.02)
    ax_top.set_ylabel('R²')
    ax_top.legend(fontsize=7, ncol=2, loc='lower right')
    ax_top.set_title(f'{hmm_name} — R² vs layer (full Bayesian beliefs)', fontweight='bold')
    ax_top.tick_params(labelbottom=False)

    # Diagonal break marks
    d = 0.015
    kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, lw=1)
    ax_top.plot((-d, +d), (-d, +d), **kwargs)
    ax_top.plot((1-d, 1+d), (-d, +d), **kwargs)
    kwargs['transform'] = ax_bot.transAxes
    ax_bot.plot((-d, +d), (1-d, 1+d), **kwargs)
    ax_bot.plot((1-d, 1+d), (1-d, 1+d), **kwargs)

    # Bottom: controls (averaged over params)
    for tname, color, ls, label in [
        ('shuffle', '#d62728', '--', 'Shuffle control'),
        ('random', '#9467bd', ':', 'Random control'),
    ]:
        sub = grp[grp['target'] == tname]
        stats = sub.groupby('layer')['R2'].agg(['mean', 'sem']).reset_index()
        ax_bot.plot(stats['layer'], stats['mean'], ls, color=color, lw=2, label=label)
        ax_bot.fill_between(stats['layer'],
                            stats['mean'] - 1.96 * stats['sem'],
                            stats['mean'] + 1.96 * stats['sem'],
                            alpha=0.15, color=color)

    ax_bot.set_ylim(ctrl_min, ctrl_max)
    ax_bot.set_xlabel('Layer')
    ax_bot.set_ylabel('R²')
    ax_bot.legend(fontsize=9)

    # Hide spines at the break
    ax_top.spines['bottom'].set_visible(False)
    ax_bot.spines['top'].set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
# Print summary: peak R² and control levels per HMM
for hmm_name, grp in r2_df.groupby('hmm'):
    real = grp[grp['target'] == 'real']
    shuf = grp[grp['target'] == 'shuffle']
    rand = grp[grp['target'] == 'random']

    peak = real.groupby('layer')['R2'].mean().max()
    peak_layer = real.groupby('layer')['R2'].mean().idxmax()
    shuf_mean = shuf['R2'].mean()
    rand_mean = rand['R2'].mean()

    print(f'{hmm_name:>8}: peak R²={peak:.4f} at L{peak_layer}, '
          f'shuffle={shuf_mean:.4f}, random={rand_mean:.4f}')


## Simplex visualizations (probe predictions)

In [ ]:
# ---- Simplex visualization: probe predictions per param per HMM ----

def barycentric_to_2d(beliefs):
    """Project 3D beliefs onto 2D equilateral triangle."""
    v1 = np.array([0, 0]); v2 = np.array([1, 0]); v3 = np.array([0.5, np.sqrt(3)/2])
    return beliefs @ np.array([v1, v2, v3])

def draw_simplex(ax):
    """Draw equilateral triangle outline."""
    verts = np.array([[0,0],[1,0],[0.5,np.sqrt(3)/2],[0,0]])
    ax.plot(verts[:,0], verts[:,1], 'k-', lw=0.8)
    ax.set_aspect('equal')
    ax.axis('off')

def extract_acts_single(input_ids, layer):
    """Forward pass, extract activations at one layer for late positions."""
    seq_len = input_ids.shape[1]
    past_kv = None
    acts_list = []

    pos_indices_g, _ = match_positions(input_ids, tok_ids_g)
    n_matched = min(seq_len, len(pos_indices_g))
    late_mask = np.arange(n_matched) >= PROBE_START
    late_model_pos = pos_indices_g[:n_matched][late_mask]

    first_hooked_chunk = None
    for start in range(0, seq_len, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, seq_len)
        chunk = input_ids[:, start:end].to(device)
        need_hooks = (end > PROBE_START)
        chunk_act = {}
        hooks = []

        if need_hooks:
            if first_hooked_chunk is None:
                first_hooked_chunk = start
            def make_hook(li):
                def fn(module, inp, out):
                    h = out[0] if isinstance(out, tuple) else out
                    chunk_act[li] = h[0]
                return fn
            try:
                hooks.append(backbone.layers[layer].register_forward_hook(make_hook(layer)))
            except:
                hooks.append(model.model.layers[layer].register_forward_hook(make_hook(layer)))

        with torch.no_grad():
            try:
                out = backbone(chunk, past_key_values=past_kv, use_cache=True)
            except:
                out = model.model(chunk, past_key_values=past_kv, use_cache=True)

        for h in hooks:
            h.remove()

        if need_hooks and layer in chunk_act:
            chunk_positions = np.arange(start, end)
            needed = np.isin(chunk_positions, late_model_pos)
            if needed.any():
                idx = np.where(needed)[0]
                acts_list.append(chunk_act[layer][idx].float().cpu().numpy())

        past_kv = out.past_key_values
        del out; torch.cuda.empty_cache()

    del past_kv; torch.cuda.empty_cache()
    return np.concatenate(acts_list, axis=0), int(late_mask.sum())

# Load R² CSV to find best layer per (hmm, param)
r2_df = pd.read_csv(os.path.join(RESULTS_DIR, 'r2_qwen35_9b.csv'))
r2_real = r2_df[r2_df['target'] == 'real']
best_layers = r2_real.groupby(['hmm', 'param']).apply(
    lambda g: g.groupby('layer')['R2'].mean().idxmax()).to_dict()

for hmm_name, cfg in HMMS.items():
    token_names = cfg['token_names']
    n_tok = len(token_names)
    tok_ids_g = [tokenizer.encode(f' {n}', add_special_tokens=False)[-1] for n in token_names]
    n_states = cfg['fn'](*cfg['params'][0])[0].shape[0]

    for param in cfg['params']:
        label = cfg['label_fn'](param)
        best_layer = int(best_layers.get((hmm_name, label), LAYERS[len(LAYERS)//4]))

        T_real = cfg['fn'](*param)
        T_stack = np.stack(T_real)
        pi_real = stationary_distribution(T_real)

        tokens = sample_hmm_sequence(T_real, pi_real, SEQ_LEN, seed=0)
        beliefs = full_bayesian_beliefs_numba(tokens.astype(np.int64), T_stack, pi_real)

        prompt = tokens_to_prompt(tokens, token_names)
        input_ids = tokenize_prompt(prompt)

        acts, n_late = extract_acts_single(input_ids, best_layer)
        y = beliefs[PROBE_START:PROBE_START + n_late]

        # Train probe (OLS with bias)
        X = np.hstack([acts, np.ones((len(acts), 1))])
        W = np.linalg.pinv(X) @ y
        preds = X @ W

        # Clip to simplex
        preds = np.clip(preds, 0, None)
        preds = preds / preds.sum(axis=1, keepdims=True)

        if n_states == 3:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            # True beliefs
            xy_true = barycentric_to_2d(y)
            draw_simplex(axes[0])
            axes[0].scatter(xy_true[:, 0], xy_true[:, 1], c=y, s=1, alpha=0.3)
            axes[0].set_title('Ground truth', fontsize=10)

            # Probe predictions
            xy_pred = barycentric_to_2d(preds)
            draw_simplex(axes[1])
            axes[1].scatter(xy_pred[:, 0], xy_pred[:, 1], c=preds, s=1, alpha=0.3)
            axes[1].set_title(f'Probe predictions (L{best_layer})', fontsize=10)

            fig.suptitle(f'{hmm_name} ({label}) — Qwen 3.5 9B', fontweight='bold')
            plt.tight_layout()
            plt.show()
            plt.close()
        else:
            # 4+ states: PCA projection
            from sklearn.decomposition import PCA
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            pca = PCA(n_components=2).fit(y)
            xy_true = pca.transform(y)
            axes[0].scatter(xy_true[:, 0], xy_true[:, 1], c=y[:, :3], s=1, alpha=0.3)
            axes[0].set_title('Ground truth (PCA)', fontsize=10)

            xy_pred = pca.transform(preds)
            axes[1].scatter(xy_pred[:, 0], xy_pred[:, 1], c=preds[:, :3], s=1, alpha=0.3)
            axes[1].set_title(f'Probe predictions (L{best_layer}, PCA)', fontsize=10)

            fig.suptitle(f'{hmm_name} ({label}) — Qwen 3.5 9B', fontweight='bold')
            plt.tight_layout()
            plt.show()
            plt.close()

        del acts; gc.collect(); torch.cuda.empty_cache()
        print(f'  {hmm_name} {label} done (layer {best_layer})')
